Data Pipeline & ASX200 Universe Audit

This notebook handles the extraction, validation and unification of price history for the ASX 200 universe. 
It identifies the data gaps (from <3 years IPO or Ticker Changes) and bridges historical data from DatAnalysis (Morningstar) with live yfinance data 

In [1]:
#IMPORTS & FOLDER SET UP  
import yfinance as yf
import pandas as pd
import os
import time
from datetime import datetime, timedelta

# Create directory structure
folders = ['data_downloads', 'manual_downloads']
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)

# Target start date for a 3-year lookback
TARGET_START = datetime.now() - timedelta(days=3*365)


The list of ASX200 Companies were taken from MarketIndex.com (https://www.marketindex.com.au/asx200) and downloaded as a CSV. 

The names were scrapped and suspended companies were manually removed - resulting in this list of 198 stocks as of 05/05/2026

In [2]:
#ASX200 NAME SCRAPPER 
df = pd.read_csv('asx200.csv')

# Pull first column, remove empty rows, and ensure only unique tickers
tickers = df.iloc[:, 0].dropna().unique().tolist()

clean_tickers = [str(t).strip().upper() for t in tickers if len(str(t).strip()) > 0]
clean_tickers.sort()

print(f"Total Unique Tickers found: {len(clean_tickers)}")
print("-" * 30)
print(clean_tickers)


Total Unique Tickers found: 198
------------------------------
['360', '4DX', 'A2M', 'AAI', 'AFI', 'AGL', 'AIA', 'ALD', 'ALK', 'ALL', 'ALQ', 'ALX', 'AMC', 'AMP', 'ANN', 'ANZ', 'APA', 'APE', 'ARG', 'ASB', 'ASK', 'ASX', 'AUB', 'AUI', 'AZJ', 'BEN', 'BFL', 'BGL', 'BHP', 'BOQ', 'BPT', 'BRG', 'BSL', 'BWP', 'BXB', 'CAR', 'CBA', 'CBO', 'CDA', 'CEN', 'CGF', 'CHC', 'CIA', 'CIP', 'CLW', 'CMM', 'CNU', 'COH', 'COL', 'CPU', 'CQR', 'CSC', 'CSL', 'CWY', 'DBI', 'DNL', 'DOW', 'DRO', 'DRR', 'DVP', 'DXS', 'DYL', 'EBO', 'EDV', 'ELV', 'EMR', 'EOS', 'EVN', 'EVT', 'FBU', 'FLT', 'FMG', 'FPH', 'FRW', 'GGP', 'GLF', 'GMD', 'GMG', 'GNE', 'GPT', 'GQG', 'GYG', 'HDN', 'HUB', 'HVN', 'IAG', 'IFT', 'IGO', 'ILU', 'IMD', 'JBH', 'JHX', 'L1G', 'LLC', 'LNW', 'LOV', 'LSF', 'LTR', 'LYC', 'MAQ', 'MCY', 'MEZ', 'MFF', 'MFG', 'MGR', 'MIN', 'MND', 'MPL', 'MQG', 'MSB', 'MTS', 'MXT', 'NAB', 'NEM', 'NHC', 'NHF', 'NIC', 'NST', 'NWH', 'NWL', 'NXG', 'NXT', 'OBM', 'ORG', 'ORI', 'PDI', 'PDN', 'PLS', 'PME', 'PMV', 'PNI', 'PPT', 'PRN', 'PRU'

In [3]:
# FULL CONSTITUENT LIST (MAY 2026) - CTD and NSR suspended at 05/05/2026
asx_200_full = [
    '360', '4DX', 'A2M', 'AAI', 'AFI', 'AGL', 'AIA', 'ALD', 'ALK', 'ALL',
    'ALQ', 'ALX', 'AMC', 'AMP', 'ANN', 'ANZ', 'APA', 'APE', 'ARG', 'ASB',
    'ASK', 'ASX', 'AUB', 'AUI', 'AZJ', 'BEN', 'BFL', 'BGL', 'BHP', 'BOQ',
    'BPT', 'BRG', 'BSL', 'BWP', 'BXB', 'CAR', 'CBA', 'CBO', 'CDA', 'CEN',
    'CGF', 'CHC', 'CIA', 'CIP', 'CLW', 'CMM', 'CNU', 'COH', 'COL', 'CPU',
    'CQR', 'CSC', 'CSL', 'CWY', 'DBI', 'DNL', 'DOW', 'DRO', 'DRR', 'DVP', 
    'DXS', 'DYL', 'EBO', 'EDV', 'ELV', 'EMR', 'EOS', 'EVN', 'EVT', 'FBU',
    'FLT', 'FMG', 'FPH', 'FRW', 'GGP', 'GLF', 'GMD', 'GMG', 'GNE', 'GPT',
    'GQG', 'GYG', 'HDN', 'HUB', 'HVN', 'IAG', 'IFT', 'IGO', 'ILU', 'IMD',
    'JBH', 'JHX', 'L1G', 'LLC', 'LNW', 'LOV', 'LSF', 'LTR', 'LYC', 'MAQ',
    'MCY', 'MEZ', 'MFF', 'MFG', 'MGR', 'MIN', 'MND', 'MPL', 'MQG', 'MSB',
    'MTS', 'MXT', 'NAB', 'NEM', 'NHC', 'NHF', 'NIC', 'NST', 'NWH', 'NWL',
    'NXG', 'NXT', 'OBM', 'ORG', 'ORI', 'PDI', 'PDN', 'PLS', 'PME', 'PMV',
    'PNI', 'PPT', 'PRN', 'PRU', 'PXA', 'QAN', 'QBE', 'QUB', 'REA', 'REG',
    'REH', 'RGN', 'RHC', 'RIO', 'RMD', 'RMS', 'RRL', 'RSG', 'RWC', 'RYM',
    'S32', 'SCG', 'SDF', 'SEK', 'SFR', 'SGH', 'SGM', 'SGP', 'SHL', 'SIG',
    'SMR', 'SOL', 'SPK', 'SRG', 'SRL', 'STO', 'SUL', 'SUN', 'TAH', 'TCL',
    'TLC', 'TLS', 'TLX', 'TNE', 'TPG', 'TUA', 'TWE', 'VAU', 'VCX', 'VEA',
    'VGN', 'VNT', 'VUL', 'WAF', 'WAM', 'WBC', 'WDS', 'WES', 'WGX', 'WHC',
    'WLE', 'WOR', 'WOW', 'WTC', 'XRO', 'XYZ', 'YAL', 'ZIP'
]

In [4]:
#AUDIT & DOWNLOAD FUNCTION
def final_asx200_audit(ticker_list, folder="data_downloads"):
    if not os.path.exists(folder): os.makedirs(folder)
    
    unfound = []
    success_count = 0
    target_start = datetime.now() - timedelta(days=3*365)
    
    print(f"--- Starting Final 200 Audit ---")

    for i, ticker in enumerate(ticker_list):
        if i % 15 == 0: time.sleep(1) #yfinance rate limit protection

        symbol = f"{ticker}.AX"
        try:
            df = yf.download(symbol, period="3y", auto_adjust=True, progress=False)
            
            if df.empty:
                print(f"❌ {ticker}: NO DATA FOUND")
                unfound.append({"ticker": ticker, "reason": "No data on Yahoo"})
                continue

            first_date = df.index.min().to_pydatetime().replace(tzinfo=None)
            
            # Gap detection for bridging
            if first_date <= target_start and len(df) >= 756: # 3 years of data (252 trading days/year)
                #Saving 'good' data to file 
                df[df.index >= target_start].to_csv(os.path.join(folder, f"{ticker}.csv"))
                success_count += 1
            else:
                reason = f"BRIDGE NEEDED: Starts {first_date.date()}"
                print(f"❌ {ticker}: {reason}")
                unfound.append({"ticker": ticker, "reason": reason})
                
        except Exception as e:
            unfound.append({"ticker": ticker, "reason": f"Error: {e}"})

    pd.DataFrame(unfound).to_csv("MANUAL_STOCK_DOWNLOAD_LIST.csv", index=False)
    print(f"\n--- AUDIT COMPLETE ---\n✅ Clean: {success_count}\n❌ Gaps/Bridge Needed: {len(unfound)}")

if __name__ == "__main__":
    final_asx200_audit(asx_200_full)


--- Starting Final 200 Audit ---
❌ AAI: BRIDGE NEEDED: Starts 2024-07-24
❌ ASK: BRIDGE NEEDED: Starts 2023-08-01
❌ CSC: BRIDGE NEEDED: Starts 2024-04-08
❌ DNL: BRIDGE NEEDED: Starts 2025-03-24
❌ ELV: BRIDGE NEEDED: Starts 2025-09-22
❌ FRW: BRIDGE NEEDED: Starts 2023-10-10
❌ GGP: BRIDGE NEEDED: Starts 2025-06-24
❌ GLF: BRIDGE NEEDED: Starts 2025-07-03
❌ GYG: BRIDGE NEEDED: Starts 2024-06-20
❌ L1G: BRIDGE NEEDED: Starts 2025-10-03
❌ LNW: BRIDGE NEEDED: Starts 2023-05-26
❌ LSF: BRIDGE NEEDED: Starts 2023-05-08
❌ NEM: BRIDGE NEEDED: Starts 2023-10-27
❌ RYM: BRIDGE NEEDED: Starts 2025-10-06
❌ SGH: BRIDGE NEEDED: Starts 2024-11-11
❌ VGN: BRIDGE NEEDED: Starts 2025-06-24
❌ XYZ: BRIDGE NEEDED: Starts 2025-01-13

--- AUDIT COMPLETE ---
✅ Clean: 181
❌ Gaps/Bridge Needed: 17


We recieved 181 'clean' stock, 17 gaps. Upon manual inspection, the gaps were categoried under the following:
1. Name Changes - Bridgeable 
    DNL (IPL), ELV (SYA), SGH (SVW) & XYZ (SQ2)
2. Foreign Dual Listed - Bridgeable 
    CSC(TSX), FRW(NZ), RYM(NZ)
3. Foreign Moved, Delisted - Bridgeable
    LNW (NASDAQ - delisted 2025, ASX data start 26/05/26)
4. New IPOs - Not Bridgeable 
    GGP, GLF, GYG & VGN (came back from 5 year hiatus)
5. M&As, etc.  - Not Bridgeable 
    AAI, ASK, L1G, NEM 

New Audit: 189 'clean' stocks, 11 gaps 

In order to bridge the stocks that are dual-listed on foreign stock exchanges, we used the code below. Currency conversion will not be done here, as the metrics the stock screener uses rely on the Z-Scores anyways. Additionally, we are downloading LNW.AX with a shorter time requirement (by the time this project is complete, the timeframe will reach 3 years of sufficient ASX data).

In [6]:
# 1. THE LNW EXCEPTION (ASX DATA)
def download_lnw_asx(folder="data_downloads"):
    if not os.path.exists(folder): os.makedirs(folder)
    ticker = "LNW.AX"
    start_date = "2023-05-26" #explicit start date to bypass the 16-day gap issue
    
    print(f"--- Downloading {ticker} ---")
    try:
        df = yf.download(ticker, start=start_date, auto_adjust=True, progress=False)
        
        if not df.empty:
            out_path = os.path.join(folder, "LNW.csv")
            df.to_csv(out_path)
            # We lower the row count to 700 to account for the May 22 start
            if len(df) >= 700:
                print(f"✅ LNW: Saved as LNW.csv (Rows: {len(df)})")
            else:
                print(f"⚠️ LNW: Saved, but rows ({len(df)}) are low. Check manually.")
        else:
            print(f"❌ LNW: No data found for {ticker} since {start_date}")
    except Exception as e:
        print(f"❌ LNW Error: {e}")


# 2. FOREIGN DUAL-LISTED 
FOREIGN_MAP = {
    "CSC": "CS.TO",    
    "FRW": "FRW.NZ",    
    "RYM": "RYM.NZ",    
}

def download_foreign_as_asx(bridge_map, folder="data_downloads"):
    if not os.path.exists(folder): os.makedirs(folder)
    
    success_count = 0
    target_start = datetime.now() - timedelta(days=3*365)
    
    print(f"\n--- Downloading Foreign Equivalents  ---")

    for asx_ticker, foreign_symbol in bridge_map.items():
        try:
            df = yf.download(foreign_symbol, period="3y", auto_adjust=True, progress=False)
            
            if df.empty:
                print(f"❌ {asx_ticker}: No data on Yahoo for {foreign_symbol}")
                continue

            first_date = df.index.min().to_pydatetime().replace(tzinfo=None)
            
            if first_date <= target_start and len(df) >= 700: 
                out_path = os.path.join(folder, f"{asx_ticker}.csv")
                df[df.index >= target_start].to_csv(out_path)
                success_count += 1
                print(f"✅ {asx_ticker}: Successfully saved {foreign_symbol} data.")
            else:
                print(f"❌ {asx_ticker}: {foreign_symbol} failed 3y check (Starts {first_date.date()}, Rows: {len(df)})")
                
        except Exception as e:
            print(f"❌ {asx_ticker}: Error {e}")

    print(f"\n--- COMPLETE ---\n✅ Foreign Stocks Added to ASX Folder: {success_count}")

if __name__ == "__main__":
    download_lnw_asx()
    download_foreign_as_asx(FOREIGN_MAP)

--- Downloading LNW.AX ---
✅ LNW: Saved as LNW.csv (Rows: 748)

--- Downloading Foreign Equivalents  ---
✅ CSC: Successfully saved CS.TO data.
✅ FRW: Successfully saved FRW.NZ data.
✅ RYM: Successfully saved RYM.NZ data.

--- COMPLETE ---
✅ Foreign Stocks Added to ASX Folder: 3


Bridging DatAnalysis Data and yfinance Data - Master Merger

The following code aligns the data from the two platform formats (keeping only OHLCV, standardises date, cleans volume, removes commas, etc.) 

Then merges the 185 yfianance files with the 4 DatAnalysis files to create one large dataset for the 02_similarity_engine

In [ ]:
#MASTER MERGER (AND CLEANER)
def simple_master_merger(raw_folder='data_downloads', manual_folder='manual_downloads'):
    master_list = []
    
    # 1. Process Automated Files (yfinance)
    raw_files = [f for f in os.listdir(raw_folder) if f.endswith('.csv')]
    for file in raw_files:
        ticker = file.replace('.csv', '')
        try:
            # Data starts at row 3
            df = pd.read_csv(
                os.path.join(raw_folder, file),
                header=0,  
                skiprows=[1, 2]  
            df = df.rename(columns={df.columns[0]: 'Date'})

            df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
            df['ticker'] = ticker
            master_list.append(df)

        except Exception as e:
            print(f"❌ {ticker}: {e}")

    # 2. Process Manual Files (DatAnalysis)
    manual_files = [f for f in os.listdir(manual_folder) if f.endswith('.csv')]
    for file in manual_files:
        ticker = file.replace('.csv', '')
        try:
            df = pd.read_csv(os.path.join(manual_folder, file))
            df = df[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']].copy()
            df['ticker'] = ticker
            master_list.append(df)

        except Exception as e:
            print(f"❌ {ticker}: {e}")

    # 3. Unify, clean, sort
    final_df = pd.concat(master_list, ignore_index=True)

    # Standardise date format
    final_df['Date'] = pd.to_datetime(final_df['Date'], format='mixed', dayfirst=True, errors='coerce')
    final_df = final_df.dropna(subset=['Date'])

    # Remove any Unnamed columns
    final_df = final_df.loc[:, ~final_df.columns.str.contains('^Unnamed')]

    # Clean Volume (DatAnalysis has commas in numbers)
    final_df['Volume'] = final_df['Volume'].astype(str).str.replace(',', '').str.replace('.00', '')
    final_df['Volume'] = pd.to_numeric(final_df['Volume'], errors='coerce')

    # Ensure price columns are numeric
    for col in ['Open', 'High', 'Low', 'Close']:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce')

    final_df = final_df.sort_values(['ticker', 'Date']).reset_index(drop=True)

    final_df.to_csv("ASX200_MASTER_DATASET.csv", index=False)
    print(f"✅ Master Dataset created: {final_df['ticker'].nunique()} tickers, {len(final_df):,} rows")
    print(f"\nSample:\n{final_df.head(10).to_string()}")
    print(f"\nDtypes:\n{final_df.dtypes}")
    print(f"\nDate range: {final_df['Date'].min().date()} → {final_df['Date'].max().date()}")

if __name__ == "__main__":
    simple_master_merger()

✅ Master Dataset created: 189 tickers, 143,581 rows

Sample:
        Date  Open  High   Low  Close    Volume ticker
0 2023-01-06  6.90  6.90  6.65   6.69  554886.0    360
1 2023-01-08  7.70  7.74  7.60   7.64  283900.0    360
2 2023-01-09  9.30  9.35  9.03   9.20  300023.0    360
3 2023-01-11  7.72  7.84  7.47   7.47  442280.0    360
4 2023-01-12  7.69  7.69  7.54   7.67  216895.0    360
5 2023-02-06  6.81  6.94  6.79   6.84  651405.0    360
6 2023-02-08  7.56  7.71  7.51   7.63  625395.0    360
7 2023-02-10  8.24  8.45  8.21   8.36  268567.0    360
8 2023-02-11  7.54  7.88  7.53   7.81  260457.0    360
9 2023-03-07  7.76  7.84  7.53   7.72  901367.0    360

Dtypes:
Date      datetime64[us]
Open             float64
High             float64
Low              float64
Close            float64
Volume           float64
ticker               str
dtype: object

Date range: 2023-01-03 → 2026-12-03
